# Dataset analysis

**Data Analytics by Axlero — SupplyPrescript**

Break the shipment history into **Train : Data validation : Testing = 60 : 20 : 20**.

| Split | Role | Share |
|---|---|---|
| Train | Fit the delay model | 60% |
| Data validation | Tune thresholds / check fit | 20% |
| Testing | Final hold-out metrics | 20% |

When `shipment_date` is present the split is **temporal** (earlier → later), matching how the model will be used on future shipments.

Split CSV files are written under `data/` and are **gitignored** so they are not pushed to GitHub.

Optional local folder for your machine:
`C:\Users\ACER\Desktop\Data Analytics by Axlero`

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from week1.split_dataset import split_frame, summarize_split, TRAIN_PATH, VAL_PATH, TEST_PATH, SUMMARY_PATH

DATA = ROOT / "data" / "shipments.csv"
assert DATA.exists(), "Run: python week1/generate_mock_data.py"

df = pd.read_csv(DATA)
print("Full dataset:", df.shape)
df.head()

In [ ]:
train, val, test, strategy = split_frame(df)
summary = summarize_split(train, val, test, strategy)

TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
train.to_csv(TRAIN_PATH, index=False)
val.to_csv(VAL_PATH, index=False)
test.to_csv(TEST_PATH, index=False)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print("=== Dataset analysis ===")
print(f"Strategy: {strategy}")
print(f"Train            : {len(train):,} rows -> {TRAIN_PATH.name}")
print(f"Data validation  : {len(val):,} rows -> {VAL_PATH.name}")
print(f"Testing          : {len(test):,} rows -> {TEST_PATH.name}")
summary

In [ ]:
# Date windows (temporal split) — train should end before validation, validation before test
for name, part in (("Train", train), ("Data validation", val), ("Testing", test)):
    if "shipment_date" in part.columns:
        dates = pd.to_datetime(part["shipment_date"])
        print(f"{name:18} {dates.min().date()} .. {dates.max().date()}  (n={len(part):,})")
    else:
        print(f"{name:18} n={len(part):,} (no shipment_date)")

## Takeaways

1. **60 : 20 : 20** — Train / Data validation / Testing.
2. **Temporal order** — earlier shipments train; newest shipments test.
3. **No GitHub data noise** — `data/*.csv` and split summaries stay local (gitignored).
4. Next: `python week1/train_model.py` uses the same 60:20:20 temporal strategy inside `DelayModel.fit()`.